# Retornos y Feature Engineering

## 1. Transformación Base: Log Returns y Retornos Simples

### Contexto y Fundamento Técnico
Antes de construir cualquier factor cuantitativo, es fundamental transformar la serie de precios brutos ($Close$ / $Adj Close$) en su representación estacionaria: los retornos. Los precios de los activos presentan no-estacionariedad y tendencias estocásticas, lo que impide usarlos directamente en modelos estadísticos.

En este bloque transformamos la matriz de precios limpios en dos tipos de rendimientos primarios:
* **Log Returns (Retornos Logarítmicos):** Utilizados para el cálculo de volatilidad, aditividad temporal a lo largo del tiempo ($\ln(P_t) - \ln(P_{t-1})$) y modelado estadístico.
* **Simple Returns (Retornos Discretos):** Utilizados para reflejar el P&L real de las estrategias, agregación transversal (cross-sectional) de activos en el portafolio y cálculo de factores de iliquidez.

> **Propósito:** Esta capa no se considera un factor en sí misma, sino la materia prima matemática a partir de la cual se derivará toda la biblioteca de señales de inversión.

## 2. Factores de Momentum y Reversión

### Contexto y Fundamento Técnico
El efecto *Momentum* es una de las anomalías de mercado más documentadas en la literatura cuantitativa (Jegadeesh & Titman, 1993). Establece que los activos que han mostrado un rendimiento superior en el pasado reciente tienden a seguir superando al mercado en el corto/medio plazo, y viceversa.

Para capturar la estructura temporal de esta anomalía sin sesgar la señal, aislamos dos dinámicas complementarias pero conceptualmente opuestas:

* **12-1 Momentum (Momentum Intermedio):** Calcula el rendimiento acumulado desde el mes $t-12$ hasta el mes $t-1$. **¿Por qué excluimos el último mes ($t-1$)?** Para evitar la contaminación por el efecto de *microestructura de mercado* y la reversión a muy corto plazo.
* **1-Month Reversal (Reversión a Corto Plazo):** Mide exclusivamente el retorno del último mes ($t-1$). Captura el exceso de reacción (*overreaction*) de los inversores a corto plazo y los desequilibrios temporales de oferta/demanda.

> **Propósito:** Generar señales de tendencia robustas separando el impulso de medio plazo de la fricción/reversión de corto plazo.

## 3. Factores de Riesgo, Volatilidad Asimétrica y Anomalía Low-Vol

### Contexto y Fundamento Técnico
La volatilidad histórica no afecta al inversor de la misma manera cuando el mercado sube que cuando cae (efecto apalancamiento o *leverage effect*). Limitarse a la desviación estándar tradicional oculta la naturaleza del riesgo al que está expuesto el portafolio.

En este bloque desagregamos el riesgo en tres métricas clave y derivamos la anomalía de baja volatilidad:

* **Rolling Volatility (252 días):** La variabilidad anualizada total de los retornos logarítmicos.
* **Downside Volatility vs. Upside Volatility:** Descomposición de la semivarianza. La *Downside Volatility* mide exclusivamente la dispersión en días de rendimiento negativo (el riesgo real de pérdida), mientras que la *Upside Volatility* mide la dispersión en días de ganancias.
* **Factor Low Volatility ($-\sigma_{252}$):** Estructuración explícita de la conocida *Low Volatility Anomaly* (Black, 1972), la cual demuestra históricamente que los portafolios de menor volatilidad obtienen retornos ajustados por riesgo superiores a los previstos por el CAPM.

> **Propósito:** Cuantificar la asimetría del riesgo en caídas vs. subidas y habilitar filtros de preservación de capital.

## 4. Factores de Liquidez (Medida de Amihud)

### Contexto y Fundamento Técnico
La liquidez es una dimensión de riesgo sistemático crucial: las acciones ilíquidas suelen requerir una prima de rentabilidad exigida por el mercado. Dado que operamos exclusivamente con datos de volumen y precio (OHLCV) sin necesidad de datos de nivel 2 (order book), la métrica académica estándar de referencia es el **Índice de Iliquidez de Amihud (2002)**.

$$K_t = \frac{|R_t|}{Dollar Volume_t}$$

* **Interpretación:** Mide el impacto absoluto en el precio ($|R_t|$) generado por cada dólar negociado en el mercado. 
* **Aplicación:** Valores altos indican activos con libros de órdenes "finos" o poco profundos, donde una orden media puede desplazar drásticamente el precio.

> **Propósito:** Detectar la fricción del mercado para evitar sobreponderar acciones difíciles de ejecutar en la vida real o para explotar la prima de iliquidez en el portafolio.